# Módulo I6 · Analítica predictiva con series temporales

**Caso:** el observatorio de Mauna Loa (Hawái) mide la concentración de CO₂ atmosférico desde 1958. Un organismo público necesita un **pronóstico a 2 años** con su incertidumbre. Las técnicas son idénticas a las que usarías para prever demanda, ventas, tráfico web o consumo energético.

Objetivos:
1. Descomponer una serie en tendencia, estacionalidad y residuo.
2. Establecer **referencias ingenuas** que cualquier modelo debe batir.
3. Ajustar suavizado exponencial (Holt-Winters) y SARIMA.
4. Validar correctamente con **backtesting** (nunca mezclar futuro y pasado).

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

## 1. Datos
Los datos semanales incluidos en statsmodels (1958–2001) se agregan a mensuales y se rellenan los huecos por interpolación.

In [ ]:
co2 = sm.datasets.co2.load_pandas().data["co2"]
serie = co2.resample("MS").mean().interpolate()
serie.index.freq = "MS"
print(f"{len(serie)} meses: {serie.index[0]:%Y-%m} → {serie.index[-1]:%Y-%m}")
serie.plot(figsize=(12, 3.5), title="CO₂ en Mauna Loa (ppm)"); plt.show()

## 2. Descomposición
STL separa la serie en **tendencia** (evolución a largo plazo), **estacionalidad** (patrón anual: la vegetación del hemisferio norte absorbe CO₂ en verano) y **residuo**.

In [ ]:
stl = STL(serie, period=12).fit()
stl.plot().set_size_inches(12, 7); plt.show()

## 3. Estacionariedad y autocorrelación
Muchos modelos (ARIMA) requieren una serie **estacionaria**: media y varianza constantes. La prueba de Dickey-Fuller aumentada (ADF) tiene como H₀ "la serie no es estacionaria".

In [ ]:
for nombre, s in [("original", serie), ("1ª diferencia", serie.diff()), ("1ª dif. + dif. estacional", serie.diff().diff(12))]:
    print(f"{nombre:28s} p-valor ADF = {adfuller(s.dropna())[1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
plot_acf(serie.diff().diff(12).dropna(), lags=36, ax=axes[0])
plot_pacf(serie.diff().diff(12).dropna(), lags=36, ax=axes[1])
plt.tight_layout(); plt.show()

## 4. Partición temporal
Reservamos los **últimos 24 meses** como test. ¡Nunca uses una partición aleatoria en series temporales!

In [ ]:
H = 24
train, test = serie[:-H], serie[-H:]

def mae(pred): return np.mean(np.abs(test.values - np.asarray(pred)))
def mape(pred): return np.mean(np.abs((test.values - np.asarray(pred)) / test.values)) * 100

## 5. Referencias ingenuas

In [ ]:
ingenuo = np.repeat(train.iloc[-1], H)                              # último valor
ingenuo_estacional = np.tile(train.iloc[-12:].values, H // 12)      # mismo mes del año anterior
deriva = train.iloc[-1] + (train.iloc[-1] - train.iloc[0]) / (len(train) - 1) * np.arange(1, H + 1)
# Estacional + deriva: mismo mes del año anterior más la subida media interanual
subida_anual = (train.iloc[-12:].values - train.iloc[-24:-12].values).mean()
estacional_deriva = np.concatenate([train.iloc[-12:].values + subida_anual * (a + 1) for a in range(H // 12)])

## 6. Holt-Winters (suavizado exponencial)
Combina un nivel, una tendencia y una componente estacional que se actualizan con pesos exponencialmente decrecientes hacia el pasado.

In [ ]:
hw = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=12).fit()
pred_hw = hw.forecast(H)

## 7. SARIMA
$\text{SARIMA}(p,d,q)(P,D,Q)_{12}$: componentes autorregresivas (AR), de diferenciación (I) y de media móvil (MA), tanto regulares como estacionales. A partir del análisis de la sección 3 probamos $(1,1,1)(0,1,1)_{12}$, un modelo clásico para series de este tipo.

In [ ]:
sarima = SARIMAX(train, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
print(sarima.summary().tables[1])
fc = sarima.get_forecast(H)
pred_sarima, ic_sarima = fc.predicted_mean, fc.conf_int(alpha=0.05)

## 8. Comparación

In [ ]:
modelos = {"Ingenuo": ingenuo, "Ingenuo estacional": ingenuo_estacional, "Deriva": deriva,
           "Estacional + deriva": estacional_deriva, "Holt-Winters": pred_hw, "SARIMA": pred_sarima}
pd.DataFrame({"MAE (ppm)": {k: mae(v) for k, v in modelos.items()},
              "MAPE (%)": {k: mape(v) for k, v in modelos.items()}}).sort_values("MAE (ppm)").round(3)

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(serie["1995":], "k", label="real")
plt.plot(test.index, ingenuo_estacional, "--", label="ingenuo estacional")
plt.plot(test.index, pred_hw, label="Holt-Winters")
plt.plot(test.index, pred_sarima, label="SARIMA")
plt.fill_between(test.index, ic_sarima.iloc[:, 0], ic_sarima.iloc[:, 1], alpha=0.2, label="IC 95 % SARIMA")
plt.axvline(test.index[0], color="gray", ls=":"); plt.legend(); plt.title("Pronóstico a 24 meses"); plt.show()

## 9. Backtesting con origen móvil
Una sola partición puede ser engañosa. Simulamos cómo habría funcionado el modelo si lo hubiéramos usado cada año: entrenamos con datos hasta el año *t* y pronosticamos los 12 meses siguientes.

In [ ]:
filas = []
for corte in pd.date_range("1994-12-01", "2000-12-01", freq="12MS"):
    tr, te = serie[:corte], serie[corte + pd.DateOffset(months=1): corte + pd.DateOffset(months=12)]
    p_hw = ExponentialSmoothing(tr, trend="add", seasonal="add", seasonal_periods=12).fit().forecast(12)
    p_sa = SARIMAX(tr, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False).forecast(12)
    p_ie = tr.iloc[-12:].values
    filas.append({"origen": corte.year, "Ingenuo estacional": np.mean(np.abs(te.values - p_ie)),
                  "Holt-Winters": np.mean(np.abs(te.values - p_hw.values)),
                  "SARIMA": np.mean(np.abs(te.values - p_sa.values))})
bt = pd.DataFrame(filas).set_index("origen")
print(bt.round(3))
print("\nMAE medio:\n", bt.mean().round(3), sep="")

## 10. Diagnóstico de residuos del modelo final
Si el modelo ha capturado toda la estructura, los residuos deberían parecer ruido blanco.

In [ ]:
sarima.plot_diagnostics(figsize=(12, 6)); plt.tight_layout(); plt.show()

## Ejercicios

1. Reentrena SARIMA con **toda** la serie y genera el pronóstico para los 24 meses siguientes con su intervalo. Redacta 3 frases para un responsable político explicando qué esperar y con qué incertidumbre.
2. Prueba `ExponentialSmoothing(..., seasonal="mul")` y `trend="add", damped_trend=True`. ¿Mejora el backtesting?
3. Busca el mejor SARIMA comparando el AIC de varias combinaciones de `(p, q)` ∈ {0, 1, 2} y `(P, Q)` ∈ {0, 1}. ¿Coincide el mejor AIC con el mejor MAE de backtesting?
4. **Demanda real:** aplica el flujo completo (descomposición → referencias → modelos → backtesting) a la serie `sm.datasets.get_rdataset("AirPassengers").data` (requiere internet) o a datos de ventas de tu empresa.

In [ ]:
# TU CÓDIGO AQUÍ